# 🤟 So sánh Model A vs Model B — ASL Recognition

| | Model A (Baseline) | Model B (Cải tiến) |
|--|--|--|
| **Data** | 650 ảnh/lớp | 650 ảnh/lớp |
| **Augmentation** | Nhẹ (rotation, shift, zoom) | Mạnh (+ brightness, contrast, noise) |
| **Loss** | Categorical Crossentropy | Label Smoothing (0.1) |
| **Dropout** | 0.4 | 0.5 + thêm 1 layer |
| **Architecture** | MobileNetV2 + head đơn giản | MobileNetV2 + head sâu hơn |

> ⚠️ **Trước khi chạy:** Runtime → Change runtime type → GPU (T4)

---
# PHẦN 1: SETUP

In [12]:
# ============================================================
# BƯỚC 1.1 — Kết nối Drive + cài thư viện
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scikit-learn seaborn

import tensorflow as tf
import numpy as np
import os, zipfile, shutil
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print(f'✅ TensorFlow: {tf.__version__}')
print(f'✅ GPU: {tf.config.list_physical_devices("GPU")}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ TensorFlow: 2.20.0
✅ GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [13]:
# BƯỚC 1.2 — Giải nén (giữ nguyên cấu trúc có sẵn)
ZIP_PATH   = '/content/drive/MyDrive/AI/asl_split.zip'
EXTRACT_TO = '/content/asl_raw'
SEED       = 42

if os.path.exists(EXTRACT_TO):
    shutil.rmtree(EXTRACT_TO)

print('📦 Đang giải nén...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_TO)
print('✅ Giải nén xong!')

# DATA_DIR trỏ thẳng vào thư mục đã có sẵn train/val/test
DATA_DIR = '/content/asl_raw/asl_split'

# Lấy danh sách lớp từ thư mục train (bỏ qua file .csv)
classes = sorted([
    d for d in os.listdir(os.path.join(DATA_DIR, 'train'))
    if os.path.isdir(os.path.join(DATA_DIR, 'train', d))
])

print(f'\n🔤 Số lớp: {len(classes)} → {classes}')
for split in ['train', 'val', 'test']:
    total = sum(len(os.listdir(os.path.join(DATA_DIR, split, c))) for c in classes)
    print(f'  {split}: {total} ảnh')

print('\n✅ Dữ liệu đã chia sẵn')

📦 Đang giải nén...
✅ Giải nén xong!

🔤 Số lớp: 28 → ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'Nothing', 'O', 'P', 'Q', 'R', 'S', 'Space', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
  train: 12232 ảnh
  val: 2614 ảnh
  test: 2654 ảnh

✅ Dữ liệu đã chia sẵn


In [14]:
# ============================================================
# BƯỚC 1.3 — Tiền xử lý CLAHE toàn bộ dataset
#
# [Kỹ thuật chưa học trên lớp]
# CLAHE (Contrast Limited Adaptive Histogram Equalization):
#   - Tăng tương phản cục bộ trên kênh L của không gian LAB
#   - clipLimit=2.0: giới hạn khuếch đại tương phản
#   - tileGridSize=(8,8): chia ảnh thành 64 ô nhỏ xử lý độc lập
#
# Lý do dùng:
#   Dataset tự chụp có ánh sáng không đồng đều
#   → CLAHE giúp model nhìn rõ đường nét ngón tay hơn
# ============================================================
import cv2

def apply_clahe(img_bgr):
    lab     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_eq    = clahe.apply(l)
    lab_eq  = cv2.merge([l_eq, a, b])
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

def apply_clahe_to_folder(data_dir, classes):
    total = 0
    for split in ['train', 'val', 'test']:
        print(f'  Đang xử lý {split}...')
        for cls in classes:
            cls_path = os.path.join(data_dir, split, cls)
            if not os.path.isdir(cls_path):
                continue
            for img_name in os.listdir(cls_path):
                if not img_name.lower().endswith(('.jpg','.jpeg','.png')):
                    continue
                img_path = os.path.join(cls_path, img_name)
                img = cv2.imread(img_path)
                if img is not None:
                    cv2.imwrite(img_path, apply_clahe(img))
                    total += 1
    return total

print('🔧 Đang áp dụng CLAHE...')
total = apply_clahe_to_folder(DATA_DIR, classes)
print(f'✅ Xong! Đã xử lý {total} ảnh')

🔧 Đang áp dụng CLAHE...
  Đang xử lý train...
  Đang xử lý val...
  Đang xử lý test...
✅ Xong! Đã xử lý 17500 ảnh


---
# PHẦN 2: MODEL A — BASELINE (650 ảnh, augmentation nhẹ)

In [15]:
# ============================================================
# BƯỚC 2.1 — Cấu hình Model A
#
# Giữ nguyên như model cũ, chỉ đổi data mới (650 ảnh)
# Augmentation nhẹ: rotation, shift, zoom, horizontal_flip=False
# Loss: Categorical Crossentropy thường
# Dropout: 0.4
# ============================================================
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

# Augmentation nhẹ (giống model cũ)
train_datagen_A = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_A = train_datagen_A.flow_from_directory(
    f'{DATA_DIR}/train', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=True, seed=SEED
)
val_A = val_test_datagen.flow_from_directory(
    f'{DATA_DIR}/val', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_A = val_test_datagen.flow_from_directory(
    f'{DATA_DIR}/test', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

CLASS_NAMES = list(train_A.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print(f'✅ Số lớp: {NUM_CLASSES}')
print(f'   Train: {train_A.samples} | Val: {val_A.samples} | Test: {test_A.samples}')

Found 12232 images belonging to 28 classes.
Found 2614 images belonging to 28 classes.
Found 2654 images belonging to 28 classes.
✅ Số lớp: 28
   Train: 12232 | Val: 2614 | Test: 2654


In [16]:
# ============================================================
# BƯỚC 2.2 — Xây dựng Model A
#
# Kiến trúc giống model cũ:
#   MobileNetV2 (freeze) → GlobalAvgPool → BN → Dense(256) → Dropout(0.4) → Output
# ============================================================
def build_model_A(num_classes):
    base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)               # Dropout nhẹ hơn
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return Model(inputs, outputs)

model_A = build_model_A(NUM_CLASSES)

# Loss thường (không label smoothing)
model_A.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('✅ Model A đã sẵn sàng!')
print(f'   Tham số trainable: {sum(tf.size(w).numpy() for w in model_A.trainable_weights):,}')

✅ Model A đã sẵn sàng!
   Tham số trainable: 337,692


In [17]:
# ============================================================
# BƯỚC 2.3 — Train Model A Giai đoạn 1 (Freeze base)
# ⏱️ Ước tính: 10-15 phút
# ============================================================
callbacks_A1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('/content/drive/MyDrive/AI/model_A_phase1.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

print('🚀 Train Model A — Giai đoạn 1 (Freeze base)...')
history_A1 = model_A.fit(
    train_A, epochs=15, validation_data=val_A,
    callbacks=callbacks_A1, verbose=1
)
print('✅ Giai đoạn 1 xong!')

🚀 Train Model A — Giai đoạn 1 (Freeze base)...
Epoch 1/15
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 409ms/step - accuracy: 0.7603 - loss: 0.8947
Epoch 1: val_accuracy improved from None to 0.98125, saving model to /content/drive/MyDrive/AI/model_A_phase1.keras

Epoch 1: finished saving model to /content/drive/MyDrive/AI/model_A_phase1.keras
383/383 ━━━━━━━━━━━━━━━━━━━━ 191s 445ms/step - accuracy: 0.8788 - loss: 0.4196 - val_accuracy: 0.9813 - val_loss: 0.0696 - learning_rate: 0.0010
Epoch 2/15
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 393ms/step - accuracy: 0.9652 - loss: 0.1171
Epoch 2: val_accuracy improved from 0.98125 to 0.98278, saving model to /content/drive/MyDrive/AI/model_A_phase1.keras

Epoch 2: finished saving model to /content/drive/MyDrive/AI/model_A_phase1.keras
383/383 ━━━━━━━━━━━━━━━━━━━━ 156s 406ms/step - accuracy: 0.9639 - loss: 0.1136 - val_accuracy: 0.9828 - val_loss: 0.0460 - learning_rate: 0.0010
Epoch 3/15
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 392ms/step - accuracy: 0.9687 - loss: 0.0933


In [18]:
# ============================================================
# BƯỚC 2.4 — Fine-tune Model A Giai đoạn 2 (Unfreeze 30 layer)
# ⏱️ Ước tính: 10-15 phút
# ============================================================
base_A = model_A.layers[1]
base_A.trainable = True
fine_tune_at = len(base_A.layers) - 30
for layer in base_A.layers[:fine_tune_at]:
    layer.trainable = False

model_A.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_A2 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('/content/drive/MyDrive/AI/model_A_final.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
]

print('🚀 Train Model A — Giai đoạn 2 (Fine-tune)...')
history_A2 = model_A.fit(
    train_A, epochs=10, validation_data=val_A,
    callbacks=callbacks_A2, verbose=1
)
print('✅ Model A hoàn thành!')

🚀 Train Model A — Giai đoạn 2 (Fine-tune)...
Epoch 1/10
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 425ms/step - accuracy: 0.9110 - loss: 0.3351
Epoch 1: val_accuracy improved from None to 0.99235, saving model to /content/drive/MyDrive/AI/model_A_final.keras

Epoch 1: finished saving model to /content/drive/MyDrive/AI/model_A_final.keras
383/383 ━━━━━━━━━━━━━━━━━━━━ 195s 461ms/step - accuracy: 0.9318 - loss: 0.2430 - val_accuracy: 0.9923 - val_loss: 0.0263 - learning_rate: 1.0000e-05
Epoch 2/10
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9623 - loss: 0.1318
Epoch 2: val_accuracy improved from 0.99235 to 0.99694, saving model to /content/drive/MyDrive/AI/model_A_final.keras

Epoch 2: finished saving model to /content/drive/MyDrive/AI/model_A_final.keras
383/383 ━━━━━━━━━━━━━━━━━━━━ 158s 412ms/step - accuracy: 0.9667 - loss: 0.1152 - val_accuracy: 0.9969 - val_loss: 0.0122 - learning_rate: 1.0000e-05
Epoch 3/10
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 406ms/step - accuracy: 0.9737 - loss: 0.070

In [ ]:
# ============================================================
# BƯỚC 2.5 — Đánh giá Model A trên tập Test
# ============================================================
print('🔍 Đánh giá Model A...')
model_A_best = tf.keras.models.load_model('/content/drive/MyDrive/AI/model_A_final.keras')

y_pred_A = np.argmax(model_A_best.predict(test_A, verbose=1), axis=1)
y_true   = test_A.classes

acc_A    = np.mean(y_pred_A == y_true)
report_A = classification_report(y_true, y_pred_A, target_names=CLASS_NAMES, output_dict=True)
cm_A     = confusion_matrix(y_true, y_pred_A)

print(f'\n📊 Model A — Test Accuracy: {acc_A*100:.2f}%')
print(f'   Macro F1  : {report_A["macro avg"]["f1-score"]*100:.2f}%')
print(f'   Precision : {report_A["macro avg"]["precision"]*100:.2f}%')
print(f'   Recall    : {report_A["macro avg"]["recall"]*100:.2f}%')

# Lưu report
report_A_str = classification_report(y_true, y_pred_A, target_names=CLASS_NAMES)
with open('/content/drive/MyDrive/AI/report_model_A.txt', 'w') as f:
    f.write(f'Test Accuracy: {acc_A:.4f}\n\n{report_A_str}')
print('✅ Đã lưu report Model A!')

---
# PHẦN 3: MODEL B — CẢI TIẾN (Augmentation mạnh + Label Smoothing + Dropout mạnh hơn)

In [20]:
# ============================================================
# BƯỚC 3.1 — Cấu hình Model B
#
# Cải tiến so với Model A:
#
# [Chưa học trên lớp]
# 1. Augmentation mạnh hơn:
#    + brightness_range: thay đổi độ sáng [0.6, 1.4]
#      → Giúp model quen với nhiều điều kiện ánh sáng
#    + shear_range: biến dạng góc nghiêng
#      → Giúp model nhận diện tay ở góc hơi nghiêng
#    + channel_shift_range: thay đổi màu sắc
#      → Giúp model không bị phụ thuộc vào màu da cố định
#
# 2. Label Smoothing (0.1):
#    → Thay vì nhãn cứng [0,0,1,0,...], làm mềm thành [0.003, 0.003, 0.91, 0.003,...]
#    → Phạt sự tự tin thái quá (overconfidence)
#    → Giảm overfitting, cải thiện tổng quát hóa
#
# 3. Dropout mạnh hơn (0.5) + thêm 1 Dense layer:
#    → Tắt 50% neuron ngẫu nhiên mỗi batch
#    → Buộc model học đặc trưng phân tán, không phụ thuộc neuron cụ thể
# ============================================================

# Augmentation MẠNH cho Model B
train_datagen_B = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,           # xoay nhiều hơn (15 thay vì 10)
    width_shift_range=0.15,      # dịch chuyển nhiều hơn
    height_shift_range=0.15,
    zoom_range=0.15,
    shear_range=0.1,             # ← MỚI: biến dạng góc nghiêng
    brightness_range=[0.6, 1.4], # ← MỚI: thay đổi độ sáng mạnh
    channel_shift_range=20.0,    # ← MỚI: thay đổi màu sắc
    horizontal_flip=False        # KHÔNG lật ngang (ký hiệu tay phân biệt trái/phải)
)

train_B = train_datagen_B.flow_from_directory(
    f'{DATA_DIR}/train', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=True, seed=SEED
)
# Val và Test dùng chung datagen không augmentation
val_B  = val_test_datagen.flow_from_directory(
    f'{DATA_DIR}/val', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_B = val_test_datagen.flow_from_directory(
    f'{DATA_DIR}/test', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
print('✅ Data generators Model B sẵn sàng!')

Found 12232 images belonging to 28 classes.
Found 2614 images belonging to 28 classes.
Found 2654 images belonging to 28 classes.
✅ Data generators Model B sẵn sàng!


In [ ]:
# ============================================================
# BƯỚC 3.2 — Xây dựng Model B
#
# Kiến trúc head sâu hơn:
#   MobileNetV2 → GlobalAvgPool → BN
#   → Dense(512) → Dropout(0.5)
#   → Dense(256) → Dropout(0.3)   ← thêm 1 layer
#   → Output(softmax)
# ============================================================
def build_model_B(num_classes):
    base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu')(x)   # lớn hơn Model A
    x = layers.Dropout(0.5)(x)                    # Dropout mạnh hơn
    x = layers.Dense(256, activation='relu')(x)   # thêm 1 Dense layer
    x = layers.Dropout(0.3)(x)                    # Dropout thứ 2
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return Model(inputs, outputs)

model_B = build_model_B(NUM_CLASSES)

# Label Smoothing: làm mềm nhãn → giảm overconfidence
model_B.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
print('✅ Model B đã sẵn sàng!')
print(f'   Tham số trainable: {sum(tf.size(w).numpy() for w in model_B.trainable_weights):,}')

In [ ]:
# ============================================================
# BƯỚC 3.3 — Train Model B Giai đoạn 1
# ⏱️ Ước tính: 10-15 phút
# ============================================================
callbacks_B1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('/content/drive/MyDrive/AI/model_B_phase1.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

print('🚀 Train Model B — Giai đoạn 1 (Freeze base)...')
history_B1 = model_B.fit(
    train_B, epochs=15, validation_data=val_B,
    callbacks=callbacks_B1, verbose=1
)
print('✅ Giai đoạn 1 xong!')

🚀 Train Model B — Giai đoạn 1 (Freeze base)...
Epoch 1/15
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 486ms/step - accuracy: 0.6232 - loss: 1.8930
Epoch 1: val_accuracy improved from None to 0.95256, saving model to /content/drive/MyDrive/AI/model_B_phase1.keras

Epoch 1: finished saving model to /content/drive/MyDrive/AI/model_B_phase1.keras
383/383 ━━━━━━━━━━━━━━━━━━━━ 212s 523ms/step - accuracy: 0.7622 - loss: 1.4629 - val_accuracy: 0.9526 - val_loss: 0.8941 - learning_rate: 0.0010
Epoch 2/15
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 469ms/step - accuracy: 0.8829 - loss: 1.1237
Epoch 2: val_accuracy improved from 0.95256 to 0.97322, saving model to /content/drive/MyDrive/AI/model_B_phase1.keras

Epoch 2: finished saving model to /content/drive/MyDrive/AI/model_B_phase1.keras
383/383 ━━━━━━━━━━━━━━━━━━━━ 185s 484ms/step - accuracy: 0.8940 - loss: 1.0974 - val_accuracy: 0.9732 - val_loss: 0.8283 - learning_rate: 0.0010
Epoch 3/15
181/383 ━━━━━━━━━━━━━━━━━━━━ 1:31 455ms/step - accuracy: 0.9171 - loss: 1.030

In [ ]:
# ============================================================
# BƯỚC 3.4 — Fine-tune Model B Giai đoạn 2
# ⏱️ Ước tính: 10-15 phút
# ============================================================
base_B = model_B.layers[1]
base_B.trainable = True
fine_tune_at = len(base_B.layers) - 30
for layer in base_B.layers[:fine_tune_at]:
    layer.trainable = False

model_B.compile(
    optimizer=Adam(learning_rate=1e-5, clipnorm=1.0),  # thêm Gradient Clipping
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_B2 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('/content/drive/MyDrive/AI/model_B_final.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
]

print('🚀 Train Model B — Giai đoạn 2 (Fine-tune + Gradient Clipping)...')
history_B2 = model_B.fit(
    train_B, epochs=10, validation_data=val_B,
    callbacks=callbacks_B2, verbose=1
)
print('✅ Model B hoàn thành!')

In [ ]:
# ============================================================
# BƯỚC 3.5 — Đánh giá Model B trên tập Test
# ============================================================
print('🔍 Đánh giá Model B...')
model_B_best = tf.keras.models.load_model('/content/drive/MyDrive/AI/model_B_final.keras')

y_pred_B = np.argmax(model_B_best.predict(test_B, verbose=1), axis=1)

acc_B    = np.mean(y_pred_B == y_true)
report_B = classification_report(y_true, y_pred_B, target_names=CLASS_NAMES, output_dict=True)
cm_B     = confusion_matrix(y_true, y_pred_B)

print(f'\n📊 Model B — Test Accuracy: {acc_B*100:.2f}%')
print(f'   Macro F1  : {report_B["macro avg"]["f1-score"]*100:.2f}%')
print(f'   Precision : {report_B["macro avg"]["precision"]*100:.2f}%')
print(f'   Recall    : {report_B["macro avg"]["recall"]*100:.2f}%')

report_B_str = classification_report(y_true, y_pred_B, target_names=CLASS_NAMES)
with open('/content/drive/MyDrive/AI/report_model_B.txt', 'w') as f:
    f.write(f'Test Accuracy: {acc_B:.4f}\n\n{report_B_str}')
print('✅ Đã lưu report Model B!')

---
# PHẦN 4: SO SÁNH MODEL A vs MODEL B

In [ ]:
# ============================================================
# BƯỚC 4.1 — Bảng so sánh tổng hợp
#
# Đây là bảng quan trọng nhất cho Chương 4.2 báo cáo!
# ============================================================
print('=' * 65)
print(f'{"BẢNG SO SÁNH MODEL A vs MODEL B":^65}')
print('=' * 65)
print(f'{"Độ đo":<20} {"Model A (Baseline)":>20} {"Model B (Cải tiến)":>20}')
print('-' * 65)
print(f'{"Accuracy":<20} {acc_A*100:>19.2f}% {acc_B*100:>19.2f}%')
print(f'{"Macro F1":<20} {report_A["macro avg"]["f1-score"]*100:>19.2f}% {report_B["macro avg"]["f1-score"]*100:>19.2f}%')
print(f'{"Macro Precision":<20} {report_A["macro avg"]["precision"]*100:>19.2f}% {report_B["macro avg"]["precision"]*100:>19.2f}%')
print(f'{"Macro Recall":<20} {report_A["macro avg"]["recall"]*100:>19.2f}% {report_B["macro avg"]["recall"]*100:>19.2f}%')
print('=' * 65)

diff = (acc_B - acc_A) * 100
print(f'\n📈 Cải thiện Accuracy: {diff:+.2f}%')
if diff > 0:
    print('✅ Model B tốt hơn Model A!')
else:
    print('⚠️  Model A vẫn tốt hơn — cần điều chỉnh thêm')

In [ ]:
# ============================================================
# BƯỚC 4.2 — Biểu đồ so sánh Training History
# ============================================================
acc_A_full     = history_A1.history['accuracy']     + history_A2.history['accuracy']
val_acc_A_full = history_A1.history['val_accuracy'] + history_A2.history['val_accuracy']
acc_B_full     = history_B1.history['accuracy']     + history_B2.history['accuracy']
val_acc_B_full = history_B1.history['val_accuracy'] + history_B2.history['val_accuracy']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Model A
axes[0].plot(acc_A_full,     'b-', label='Train', linewidth=2)
axes[0].plot(val_acc_A_full, 'r--', label='Val',  linewidth=2)
axes[0].axvline(x=len(history_A1.history['accuracy']),
                color='gray', linestyle=':', label='Fine-tune')
axes[0].set_title('Model A — Baseline', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

# Model B
axes[1].plot(acc_B_full,     'b-', label='Train', linewidth=2)
axes[1].plot(val_acc_B_full, 'r--', label='Val',  linewidth=2)
axes[1].axvline(x=len(history_B1.history['accuracy']),
                color='gray', linestyle=':', label='Fine-tune')
axes[1].set_title('Model B — Cải tiến', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1.05])

plt.suptitle('So sánh quá trình Training: Model A vs Model B', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AI/compare_training_history.png', dpi=150)
plt.show()
print('✅ Đã lưu biểu đồ so sánh!')

In [ ]:
# ============================================================
# BƯỚC 4.3 — Confusion Matrix cả 2 model
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(28, 12))

for ax, cm, title, acc in [
    (axes[0], cm_A, f'Model A — Baseline\n(Acc: {acc_A*100:.1f}%)', acc_A),
    (axes[1], cm_B, f'Model B — Cải tiến\n(Acc: {acc_B*100:.1f}%)', acc_B)
]:
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cmap='Blues', linewidths=0.3)
    ax.set_title(title, fontsize=13, pad=10)
    ax.set_ylabel('Nhãn thực tế')
    ax.set_xlabel('Nhãn dự đoán')

plt.suptitle('Confusion Matrix: Model A vs Model B', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AI/compare_confusion_matrix.png', dpi=120)
plt.show()
print('✅ Đã lưu confusion matrix!')

In [ ]:
# ============================================================
# BƯỚC 4.4 — Biểu đồ cột so sánh F1 từng lớp
#
# Giúp thấy rõ lớp nào Model B cải thiện nhiều nhất
# ============================================================
f1_A = [report_A[cls]['f1-score'] for cls in CLASS_NAMES]
f1_B = [report_B[cls]['f1-score'] for cls in CLASS_NAMES]

x = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(18, 6))
bars_A = ax.bar(x - width/2, f1_A, width, label='Model A', color='steelblue', alpha=0.8)
bars_B = ax.bar(x + width/2, f1_B, width, label='Model B', color='coral',     alpha=0.8)

ax.set_xlabel('Ký hiệu tay (Lớp)', fontsize=12)
ax.set_ylabel('F1-score', fontsize=12)
ax.set_title('F1-score từng lớp: Model A vs Model B', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.legend(fontsize=11)
ax.set_ylim([0, 1.1])
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AI/compare_f1_per_class.png', dpi=150)
plt.show()
print('✅ Đã lưu biểu đồ F1 từng lớp!')

In [ ]:
# ============================================================
# BƯỚC CUỐI — Kiểm tra tất cả file đã lưu
# ============================================================
output_files = [
    '/content/drive/MyDrive/AI/model_A_final.keras',
    '/content/drive/MyDrive/AI/model_B_final.keras',
    '/content/drive/MyDrive/AI/report_model_A.txt',
    '/content/drive/MyDrive/AI/report_model_B.txt',
    '/content/drive/MyDrive/AI/compare_training_history.png',
    '/content/drive/MyDrive/AI/compare_confusion_matrix.png',
    '/content/drive/MyDrive/AI/compare_f1_per_class.png',
]

print('📦 Kiểm tra file đầu ra:')
for f in output_files:
    exists = os.path.exists(f)
    size   = os.path.getsize(f)/1024 if exists else 0
    status = f'✅ ({size:.0f} KB)' if exists else '❌'
    print(f'  {status} {os.path.basename(f)}')

print('\n🎉 Hoàn thành! Sẵn sàng viết báo cáo Chương 4.')